In [23]:
from itera_llm.model import criar_modelo
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.chains.llm import LLMChain

from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser,
)
from langchain_classic.chains.router.multi_prompt_prompt import (
    MULTI_PROMPT_ROUTER_TEMPLATE,
)

In [24]:
quimica_template = ChatPromptTemplate.from_template("""Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: {input}""")

geografia_template = ChatPromptTemplate.from_template("""Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana, 
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: {input}""")

biologia_template = ChatPromptTemplate.from_template("""Você é um biólogo muito capacitado.
Você tem um grande conhecimento sobre os seres vivos, suas estruturas, funções
e a interação entre diferentes organismos e seus ambientes.
Você é excelente em explicar conceitos de biologia de maneira clara,
tanto para iniciantes quanto para estudantes avançados.

Aqui está uma pergunta: {input}""")


In [25]:
prompt_infos = [
    {
        "name": "Química",
        "description": "Ideal para responder pergunta de química",
        "prompt_template": quimica_template
    },
     {
        "name": "Geografia",
        "description": "Ideal para responder pergunta de geografia",
        "prompt_template": geografia_template
    },
      {
        "name": "Biologia",
        "description": "Ideal para responder pergunta de biologia",
        "prompt_template": biologia_template
    },
]

In [26]:
chat = criar_modelo()

In [27]:
chains_destino = {}
for info in prompt_infos:
    chain = LLMChain(llm=chat, prompt=info["prompt_template"], verbose=True)
    chains_destino[info["name"]] = chain
    
chains_destino

{'Química': LLMChain(verbose=True, prompt=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='Você é um químico muito experiente.\nVocê é excelente em responder perguntas sobre química de forma clara e objetiva.\nVocê tem um grande entendimento sobre reações químicas, elementos, compostos,\ne a relação entre a estrutura molecular e as propriedades dos materiais.\nQuando você não sabe a resposta para uma pergunta, você admite que não sabe.\n\nAqui está uma pergunta: {input}'), additional_kwargs={})]), llm=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-6 Luna', 'release_date': '2026-09-22', 'last_updated': '2026-09-22', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs

In [28]:
destinos = [f'{p["name"]}: {p["description"]}' for p in prompt_infos]
destinos_str = "\n".join(destinos)
print(destinos_str)

Química: Ideal para responder pergunta de química
Geografia: Ideal para responder pergunta de geografia
Biologia: Ideal para responder pergunta de biologia


In [29]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations= destinos_str
)

print(router_template)

Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
Química: Ideal para responder pergunta de química
Geografia: Ideal para responder pergunta de geograf

In [30]:
router_template = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

router_chain = LLMRouterChain.from_llm(chat, router_template, verbose=True)

In [ ]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=chat, prompt=default_prompt, verbose=True)
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=chains_destino,
    default_chain=default_chain,
    verbose=True
) 

In [32]:
chain.invoke({"input": "O que é o El Niño?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...



> Finished chain.
Geografia: {'input': 'O que é o El Niño?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana, 
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: O que é o El Niño?

> Finished chain.

> Finished chain.


{'input': 'O que é o El Niño?',
 'text': 'O **El Niño** é um fenômeno climático natural que ocorre quando as águas superficiais do **oceano Pacífico equatorial central e oriental** ficam mais quentes do que o normal por vários meses. Ele faz parte do ciclo **El Niño–Oscilação Sul (ENOS)**, que também inclui a fase fria, chamada **La Niña**.\n\nEm condições normais, os ventos alísios empurram as águas quentes para o oeste do Pacífico. Durante o El Niño, esses ventos enfraquecem, permitindo que a água quente se espalhe para leste. Isso altera a circulação da atmosfera e pode mudar os padrões de chuva e temperatura em muitas regiões do planeta.\n\nOs efeitos variam conforme a intensidade e a época do fenômeno. Em algumas áreas, pode haver mais chuvas e enchentes; em outras, secas, ondas de calor ou mudanças na atividade pesqueira. No Brasil, por exemplo, o El Niño costuma favorecer mais chuva no Sul e condições mais quentes e secas em partes do Norte e do Nordeste, embora os impactos não 

In [33]:
chain.invoke({"input": "Pra que serve a tabela periódica?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
Química: {'input': 'Pra que serve a tabela periódica?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: Pra que serve a tabela periódica?

> Finished chain.

> Finished chain.


{'input': 'Pra que serve a tabela periódica?',
 'text': 'A tabela periódica organiza os elementos químicos de acordo com o número de prótons de seus átomos. Ela serve para:\n\n- **Consultar informações** sobre cada elemento, como símbolo, número atômico e massa aproximada.\n- **Agrupar elementos com propriedades semelhantes**, o que ajuda a prever como eles se comportam.\n- **Entender e prever reações químicas**: a posição de um elemento dá pistas sobre como ele pode reagir e que tipos de ligações pode formar.\n\nPor exemplo, sódio e potássio ficam na mesma coluna e têm comportamentos químicos parecidos. Assim, a tabela é uma espécie de mapa para estudar e compreender os elementos e suas combinações.'}

In [34]:
chain.invoke({"input": "Pra que serve a fotossíntese?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
Biologia: {'input': 'Pra que serve a fotossíntese?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um biólogo muito capacitado.
Você tem um grande conhecimento sobre os seres vivos, suas estruturas, funções
e a interação entre diferentes organismos e seus ambientes.
Você é excelente em explicar conceitos de biologia de maneira clara,
tanto para iniciantes quanto para estudantes avançados.

Aqui está uma pergunta: Pra que serve a fotossíntese?

> Finished chain.

> Finished chain.


{'input': 'Pra que serve a fotossíntese?',
 'text': 'A fotossíntese serve para transformar a energia da luz em energia química. Plantas, algas e algumas bactérias usam essa energia para produzir açúcares a partir de água e dióxido de carbono.\n\nDe forma simplificada:\n\n**água + dióxido de carbono + luz → açúcar + oxigênio**\n\nO açúcar fornece energia e matéria para o crescimento desses organismos. Já o oxigênio liberado é importante para a respiração de muitos seres vivos. Por isso, a fotossíntese sustenta boa parte das cadeias alimentares e ajuda a manter o equilíbrio de gases na atmosfera.'}

In [ ]:
gen = (x ** 2 for x in range(10))
print(gen)

quadrado_tuple = tuple(gen)
print(quadrado_tuple)

<generator object <genexpr> at 0x000002216F112670>


TypeError: 'generator' object is not callable